# Backtracking en Python

**Curso:** Técnicas de Programación · Unidad 2
**Tema:** Backtracking

> Este notebook desarrolla la **plantilla universal de backtracking**, ejemplos resueltos con la misma estructura
> (probar → avanzar → retroceder), una medición concreta de la **poda temprana** (nodos explorados con y sin restricciones),
> aplicaciones clásicas (N-Reinas, mochila 0-1, caminos en un laberinto) y **ejercicios propuestos** para completar.
>
> 📎 **Material de apoyo (teoría completa):** revisa primero la página [Backtracking](Backtracking.md) de esta unidad — tiene
> la comparación con fuerza bruta/recursión/PD, la analogía del laberinto y el PDF de la clase.


## Contenidos
1. [¿Qué es backtracking? (recap)](#que-es)
2. [Plantilla universal de backtracking](#plantilla)
3. [Ejemplos resueltos con la plantilla](#ejemplos)
    - Cadenas de A/B de longitud n
    - Listas de 1 a 3 de longitud n
    - Permutaciones de una lista
4. [Midiendo la poda: nodos explorados](#poda)
5. [Generación de subconjuntos (patrón Incluir/Excluir)](#subconjuntos)
6. [Poda con restricciones: N-Reinas](#nreinas)
7. [Aplicación: Mochila 0-1 por backtracking](#mochila)
8. [Aplicación: caminos en un laberinto](#laberinto)
9. [Ejercicios propuestos](#ejercicios)
10. [Soluciones de referencia (opcional)](#soluciones)
11. [Apéndice: cuándo usar backtracking, errores frecuentes y material de apoyo](#apendice)


<a id="que-es"></a>
## 1) ¿Qué es backtracking? (recap)
El **backtracking** construye una solución **incrementalmente**, probando candidatos parciales, y **retrocede** tan pronto
como detecta que un camino no puede conducir a una solución válida. Esa **poda temprana** es lo que lo diferencia de la
fuerza bruta pura (que genera todo y valida al final).

La estructura es siempre la misma, en tres fases:
1. **Probar**: elegir una opción y verificar si es válida.
2. **Avanzar**: si es válida, agregarla a la solución parcial y recurrir.
3. **Retroceder**: al volver, deshacer la opción y probar la siguiente.

> La teoría completa (espacio de estados, árbol de decisiones, comparación con otros paradigmas) está en la página
> [Backtracking](Backtracking.md) de esta unidad.


<a id="plantilla"></a>
## 2) Plantilla universal de backtracking
Los cuatro componentes que cambian según el problema son la **condición de éxito**, la **validez** de una opción, y cómo se
**agrega/deshace** una opción de la solución parcial. La arquitectura de tres fases permanece constante.


In [1]:
def backtracking_template(opciones, profundidad):
    """Plantilla universal de backtracking: genera todas las secuencias de 'profundidad'
    elementos tomados (con repeticion) de 'opciones'."""
    resultado = []

    def backtrack(solucion):
        # 1. Condicion de exito: la solucion alcanzo la profundidad deseada
        if len(solucion) == profundidad:
            resultado.append(solucion[:])   # copiamos, no referenciamos
            return

        # 2. Recorrer todas las opciones posibles
        for opcion in opciones:
            solucion.append(opcion)          # PROBAR + AVANZAR (aqui no hay restriccion que validar)
            backtrack(solucion)               # recursion
            solucion.pop()                    # RETROCEDER (deshacer)

    backtrack([])
    return resultado

# Ejemplo rapido de uso de la plantilla
print(backtracking_template(["A", "B"], 2))


[['A', 'A'], ['A', 'B'], ['B', 'A'], ['B', 'B']]


<a id="ejemplos"></a>
## 3) Ejemplos resueltos con la plantilla
Tres variaciones directas de la plantilla: solo cambian las **opciones** y cómo se **guarda** la solución.

### Cadenas de `A` y `B` de longitud `n`


In [2]:
def cadenas_AB(n):
    """Genera todas las cadenas posibles de tamano n usando las letras A y B."""
    resultado = []

    def backtrack(solucion):
        if len(solucion) == n:                    # caso base
            resultado.append(''.join(solucion))
            return
        for letra in ["A", "B"]:                   # opciones
            solucion.append(letra)                  # probar + avanzar
            backtrack(solucion)
            solucion.pop()                           # retroceder

    backtrack([])
    return resultado

print(cadenas_AB(3))


['AAA', 'AAB', 'ABA', 'ABB', 'BAA', 'BAB', 'BBA', 'BBB']


### Listas de números del 1 al 3 de longitud `n`

In [3]:
def listas_1_a_3(n):
    """Genera todas las listas de longitud n usando los numeros 1, 2 y 3."""
    resultado = []

    def backtrack(solucion):
        if len(solucion) == n:
            resultado.append(solucion[:])
            return
        for num in [1, 2, 3]:
            solucion.append(num)
            backtrack(solucion)
            solucion.pop()

    backtrack([])
    return resultado

print(listas_1_a_3(3))


[[1, 1, 1], [1, 1, 2], [1, 1, 3], [1, 2, 1], [1, 2, 2], [1, 2, 3], [1, 3, 1], [1, 3, 2], [1, 3, 3], [2, 1, 1], [2, 1, 2], [2, 1, 3], [2, 2, 1], [2, 2, 2], [2, 2, 3], [2, 3, 1], [2, 3, 2], [2, 3, 3], [3, 1, 1], [3, 1, 2], [3, 1, 3], [3, 2, 1], [3, 2, 2], [3, 2, 3], [3, 3, 1], [3, 3, 2], [3, 3, 3]]


### Permutaciones de una lista pequeña
**Problema:** dada una lista sin elementos repetidos, generar todas sus permutaciones. La opción "no está usado todavía"
es la condición de validez.


In [4]:
def permutaciones(nums):
    """Genera todas las permutaciones de nums. Complejidad: O(n! * n)."""
    resultado = []

    def backtrack(solucion):
        if len(solucion) == len(nums):             # caso base: permutacion completa
            resultado.append(solucion[:])
            return
        for num in nums:
            if num in solucion:                      # PROBAR: valido si no se ha usado (O(n))
                continue
            solucion.append(num)                       # AVANZAR
            backtrack(solucion)
            solucion.pop()                               # RETROCEDER

    backtrack([])
    return resultado

print(permutaciones([1, 2, 3]))
print(permutaciones(["A", "B", "C"]))


[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
[['A', 'B', 'C'], ['A', 'C', 'B'], ['B', 'A', 'C'], ['B', 'C', 'A'], ['C', 'A', 'B'], ['C', 'B', 'A']]


**Optimización:** `num in solucion` es `O(n)` por cada elemento probado. Con un `set` auxiliar la validación baja a
`O(1)`, y el total pasa de `O(n² · n!)` a `O(n · n!)` — notable para `n ≥ 8`.


In [5]:
def permutaciones_optimizada(nums):
    """Version optimizada: un set 'usados' valida en O(1) en vez de O(n)."""
    resultado = []
    usados = set()

    def backtrack(solucion):
        if len(solucion) == len(nums):
            resultado.append(solucion[:])
            return
        for num in nums:
            if num not in usados:
                usados.add(num)
                solucion.append(num)
                backtrack(solucion)
                solucion.pop()
                usados.remove(num)

    backtrack([])
    return resultado

assert sorted(permutaciones_optimizada([1, 2, 3])) == sorted(permutaciones([1, 2, 3]))
print("OK: permutaciones_optimizada coincide con permutaciones")


OK: permutaciones_optimizada coincide con permutaciones


<a id="poda"></a>
## 4) Midiendo la poda: nodos explorados
Generamos todas las cadenas binarias de longitud `n` (sin restricciones) y contamos cuántos **nodos** visita el árbol de
llamadas. Sin restricciones que podar, backtracking explora exactamente lo mismo que la fuerza bruta: `2^(n+1) - 1` nodos
para un árbol binario de altura `n`. La poda real la vemos en la sección de N-Reinas, donde sí hay restricciones.


In [6]:
def combinaciones_binarias(n):
    """Genera todas las cadenas binarias de longitud n. Tambien cuenta nodos visitados."""
    resultado = []
    nodos_visitados = 0

    def backtrack(solucion):
        nonlocal nodos_visitados
        nodos_visitados += 1                        # contamos este nodo (llamada)
        if len(solucion) == n:                        # caso base
            resultado.append(''.join(map(str, solucion)))
            return
        for bit in (0, 1):                             # opciones
            solucion.append(bit)
            backtrack(solucion)
            solucion.pop()

    backtrack([])
    return resultado, nodos_visitados

for n in (3, 5, 10):
    combos, nodos = combinaciones_binarias(n)
    esperado = 2**(n + 1) - 1
    assert nodos == esperado
    print(f"n={n}: {len(combos)} combinaciones, {nodos} nodos visitados (formula 2^(n+1)-1 = {esperado})")


n=3: 8 combinaciones, 15 nodos visitados (formula 2^(n+1)-1 = 15)
n=5: 32 combinaciones, 63 nodos visitados (formula 2^(n+1)-1 = 63)
n=10: 1024 combinaciones, 2047 nodos visitados (formula 2^(n+1)-1 = 2047)


<a id="subconjuntos"></a>
## 5) Generación de subconjuntos (patrón Incluir/Excluir)
**Problema:** dado un conjunto de elementos, generar todos sus subconjuntos posibles (el conjunto potencia). Para
`{1, 2, 3}` hay `2^3 = 8` subconjuntos. En cada posición decidimos **incluir** o **excluir** el elemento — cada solución
parcial (incluso incompleta) ya es un subconjunto válido, así que se registra en **cada** llamada, no solo en las hojas.

**Complejidad:** temporal `O(n · 2^n)` (2ⁿ subconjuntos, copiar cada uno cuesta `O(n)`), espacial `O(n)` (profundidad de
la recursión).


In [7]:
def subconjuntos(nums):
    """Genera todos los subconjuntos de nums (patron incluir/excluir por posicion)."""
    resultado = []

    def backtrack(inicio, subconjunto):
        resultado.append(subconjunto[:])              # cada estado parcial ya es valido

        for i in range(inicio, len(nums)):
            subconjunto.append(nums[i])                 # PROBAR + AVANZAR: incluir nums[i]
            backtrack(i + 1, subconjunto)
            subconjunto.pop()                             # RETROCEDER: excluir nums[i]

    backtrack(0, [])
    return resultado

print(subconjuntos([1, 2, 3]))


[[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


<a id="nreinas"></a>
## 6) Poda con restricciones: N-Reinas
**Problema:** colocar `N` reinas en un tablero `N×N` sin que se ataquen entre sí (misma fila, columna o diagonal). Aquí sí
hay una restricción real que evaluar en cada paso, y por lo tanto poda real: si una columna ya está ocupada o cae en una
diagonal ocupada, **ni siquiera se avanza con ella**.


In [8]:
def n_reinas(n, contar_nodos=False):
    """Devuelve todas las soluciones de colocar n reinas sin que se ataquen.
    Si contar_nodos=True, tambien devuelve cuantos nodos (llamadas) se exploraron."""
    soluciones = []
    columnas, diag1, diag2 = set(), set(), set()   # diag1: fila-col, diag2: fila+col
    nodos = 0

    def backtrack(solucion):
        nonlocal nodos
        nodos += 1
        fila = len(solucion)
        if fila == n:                                    # caso base: una reina por fila
            soluciones.append(solucion[:])
            return
        for col in range(n):
            if col in columnas or (fila - col) in diag1 or (fila + col) in diag2:
                continue                                    # PROBAR: no es valida, se poda (ni se avanza)
            columnas.add(col); diag1.add(fila - col); diag2.add(fila + col)   # AVANZAR
            solucion.append(col)
            backtrack(solucion)
            solucion.pop()                                                      # RETROCEDER
            columnas.remove(col); diag1.remove(fila - col); diag2.remove(fila + col)

    backtrack([])
    return (soluciones, nodos) if contar_nodos else soluciones


def imprimir_tablero(solucion):
    n = len(solucion)
    for fila in range(n):
        print("".join(" Q " if solucion[fila] == col else " . " for col in range(n)))
    print()


soluciones, nodos = n_reinas(4, contar_nodos=True)
print(f"N-Reinas(4): {len(soluciones)} soluciones, {nodos} nodos explorados (con poda)")
print(f"Fuerza bruta sin poda habria probado 4^4 = {4**4} tableros completos antes de validar cada uno")
for sol in soluciones:
    imprimir_tablero(sol)


N-Reinas(4): 2 soluciones, 17 nodos explorados (con poda)
Fuerza bruta sin poda habria probado 4^4 = 256 tableros completos antes de validar cada uno
 .  Q  .  . 
 .  .  .  Q 
 Q  .  .  . 
 .  .  Q  . 

 .  .  Q  . 
 Q  .  .  . 
 .  .  .  Q 
 .  Q  .  . 



<a id="mochila"></a>
## 7) Aplicación: Mochila 0-1 por backtracking
**Problema:** dado un conjunto de objetos con peso y valor, y una capacidad máxima, decidir para cada objeto si se toma
(1) o no (0), maximizando el valor sin exceder la capacidad. Es el patrón incluir/excluir aplicado a optimización: se
exploran las `2^n` combinaciones y se descarta cualquiera que exceda la capacidad.


In [9]:
def mochila_01(pesos, valores, capacidad):
    """Explora todas las combinaciones (tomar/no tomar) y retorna la de mayor valor
    que no exceda la capacidad. Fuerza bruta con poda simple por backtracking."""
    mejor_valor = 0
    mejor_solucion = []

    def backtrack(i, solucion, peso_acum, valor_acum):
        nonlocal mejor_valor, mejor_solucion

        if peso_acum > capacidad:                 # PODA: esta rama ya no puede ser valida
            return

        if i == len(pesos):                         # caso base: decidimos sobre todos los objetos
            if valor_acum > mejor_valor:
                mejor_valor = valor_acum
                mejor_solucion = solucion[:]
            return

        for tomar in (0, 1):                          # opciones: no tomar u tomar el objeto i
            solucion.append(tomar)
            nuevo_peso = peso_acum + (pesos[i] if tomar else 0)
            nuevo_valor = valor_acum + (valores[i] if tomar else 0)
            backtrack(i + 1, solucion, nuevo_peso, nuevo_valor)
            solucion.pop()

    backtrack(0, [], 0, 0)
    return mejor_valor, mejor_solucion


pesos = [2, 3, 4, 5]
valores = [3, 4, 5, 6]
capacidad = 5

mejor_valor, mejor_solucion = mochila_01(pesos, valores, capacidad)
print("Mejor valor:", mejor_valor)
print("Solucion (0=no tomar, 1=tomar):", mejor_solucion)


Mejor valor: 7
Solucion (0=no tomar, 1=tomar): [1, 1, 0, 0]


<a id="laberinto"></a>
## 8) Aplicación: caminos en un laberinto
**Problema:** dado un laberinto (matriz con `0` = libre y `1` = obstáculo), encontrar todos los caminos desde una celda
inicial hasta una final, moviéndose en las 4 direcciones sin repetir celda. La matriz `visitado` es el estado que se
**marca al avanzar y se desmarca al retroceder** — si no se desmarcara, el backtracking no podría explorar otras rutas
que pasen por la misma celda.


In [10]:
def caminos_en_laberinto(laberinto, inicio, fin):
    """Encuentra todos los caminos entre inicio y fin en un laberinto, moviendose en las 4 direcciones."""
    filas, columnas = len(laberinto), len(laberinto[0])
    visitado = [[False] * columnas for _ in range(filas)]
    movimientos = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # arriba, abajo, izquierda, derecha
    resultado = []

    def es_valida(f, c):
        return 0 <= f < filas and 0 <= c < columnas and laberinto[f][c] == 0 and not visitado[f][c]

    def backtrack(pos, camino):
        if pos == fin:                                     # caso base: llegamos a la meta
            resultado.append(camino[:])
            return
        f, c = pos
        for df, dc in movimientos:
            nf, nc = f + df, c + dc
            if es_valida(nf, nc):
                visitado[nf][nc] = True                        # AVANZAR: marcar visitada
                camino.append((nf, nc))
                backtrack((nf, nc), camino)
                camino.pop()                                       # RETROCEDER: desmarcar
                visitado[nf][nc] = False

    fi, ci = inicio
    ff, cf = fin
    if laberinto[fi][ci] == 1 or laberinto[ff][cf] == 1:
        return []                                               # inicio o fin son obstaculos

    visitado[fi][ci] = True
    backtrack(inicio, [inicio])
    return resultado


laberinto = [
    [0, 0, 0, 1],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
]

rutas = caminos_en_laberinto(laberinto, inicio=(0, 0), fin=(2, 3))
print("Numero de rutas encontradas:", len(rutas))
for ruta in rutas:
    print(ruta)


Numero de rutas encontradas: 1
[(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]


<a id="ejercicios"></a>
## 9) Ejercicios propuestos
En todos, mantén la misma estructura de backtracking:

```python
resultado = []
def backtrack(solucion):
    if condicion_de_parada:
        guardar_solucion
        return
    for opcion in opciones:
        hacer
        backtrack(...)
        deshacer
```
Solo cambian las **opciones**, la **condición de parada** y la forma de **guardar la solución**.

1. Cadenas de `A`, `B` y `C` de longitud `n`.
2. Listas de números del 1 al 3 de longitud `n`.
3. Simular lanzamientos de un dado (1-6) `n` veces.
4. Cadenas de dígitos (0-9) de longitud `n`.
5. Cadenas de `A`, `B`, `C` donde no haya dos letras iguales seguidas.
6. Secuencias no decrecientes de números del 1 al 5.
7. Todas las permutaciones de `[1, 2, 3]` (ya viste este patrón arriba — inténtalo sin mirar).
8. Todas las permutaciones de `['A', 'B', 'C']`.
9. Cadenas binarias de longitud `n` con exactamente `k` unos.
10. Combinaciones de Tomar/No tomar (`'T'`/`'N'`) para una lista de `n` elementos.


In [11]:
# Completa cada funcion usando la misma estructura de backtracking.
# Los parametros y el TODO son una guia; ajusta la firma si tu solucion lo necesita.

def ejercicio_1(n):
    resultado = []
    def backtrack(solucion):
        # TODO: condicion de parada -> if len(solucion) == n: resultado.append(...); return
        # TODO: opciones -> for letra in ["A", "B", "C"]: ...
        pass
    backtrack([])
    return resultado


def ejercicio_2(n):
    resultado = []
    def backtrack(solucion):
        # TODO
        pass
    backtrack([])
    return resultado


def ejercicio_3(n):
    resultado = []
    def backtrack(solucion):
        # TODO: opciones -> for cara in range(1, 7): ...
        pass
    backtrack([])
    return resultado


def ejercicio_4(n):
    resultado = []
    def backtrack(solucion):
        # TODO: opciones -> for digito in range(10): ...
        pass
    backtrack([])
    return resultado


def ejercicio_5(n):
    resultado = []
    def backtrack(solucion):
        # TODO: valido si solucion esta vacia o el ultimo elemento != la opcion actual
        pass
    backtrack([])
    return resultado


def ejercicio_6(maximo=5):
    resultado = []
    def backtrack(solucion):
        # TODO: opciones deben ser >= al ultimo elemento agregado (no decrecientes)
        pass
    backtrack([])
    return resultado


def ejercicio_7(nums):
    resultado = []
    def backtrack(solucion):
        # TODO: reutiliza el patron de "permutaciones" visto arriba
        pass
    backtrack([])
    return resultado


def ejercicio_8(letras):
    resultado = []
    def backtrack(solucion):
        # TODO
        pass
    backtrack([])
    return resultado


def ejercicio_9(n, k):
    resultado = []
    def backtrack(solucion, unos_usados):
        # TODO: condicion de parada -> len(solucion) == n and unos_usados == k
        pass
    backtrack([], 0)
    return resultado


def ejercicio_10(n):
    resultado = []
    def backtrack(solucion):
        # TODO: opciones -> for decision in ["T", "N"]: ...
        pass
    backtrack([])
    return resultado


print("Plantillas listas para completar: ejercicio_1 .. ejercicio_10")


Plantillas listas para completar: ejercicio_1 .. ejercicio_10


<a id="soluciones"></a>
## 10) Soluciones de referencia (opcional)
Solo dos, para que compruebes tu enfoque sin arruinar el resto de los ejercicios — inténtalos primero por tu cuenta.


In [12]:
def solucion_ejercicio_1(n):
    """Ejercicio 1: cadenas de A, B y C de longitud n."""
    resultado = []

    def backtrack(solucion):
        if len(solucion) == n:
            resultado.append(''.join(solucion))
            return
        for letra in ["A", "B", "C"]:
            solucion.append(letra)
            backtrack(solucion)
            solucion.pop()

    backtrack([])
    return resultado


def solucion_ejercicio_2(n):
    """Ejercicio 2: listas de numeros del 1 al 3 de longitud n."""
    resultado = []

    def backtrack(solucion):
        if len(solucion) == n:
            resultado.append(solucion[:])
            return
        for num in [1, 2, 3]:
            solucion.append(num)
            backtrack(solucion)
            solucion.pop()

    backtrack([])
    return resultado


print(solucion_ejercicio_1(3))
print(solucion_ejercicio_2(3))


['AAA', 'AAB', 'AAC', 'ABA', 'ABB', 'ABC', 'ACA', 'ACB', 'ACC', 'BAA', 'BAB', 'BAC', 'BBA', 'BBB', 'BBC', 'BCA', 'BCB', 'BCC', 'CAA', 'CAB', 'CAC', 'CBA', 'CBB', 'CBC', 'CCA', 'CCB', 'CCC']
[[1, 1, 1], [1, 1, 2], [1, 1, 3], [1, 2, 1], [1, 2, 2], [1, 2, 3], [1, 3, 1], [1, 3, 2], [1, 3, 3], [2, 1, 1], [2, 1, 2], [2, 1, 3], [2, 2, 1], [2, 2, 2], [2, 2, 3], [2, 3, 1], [2, 3, 2], [2, 3, 3], [3, 1, 1], [3, 1, 2], [3, 1, 3], [3, 2, 1], [3, 2, 2], [3, 2, 3], [3, 3, 1], [3, 3, 2], [3, 3, 3]]


<a id="apendice"></a>
## 11) Apéndice: cuándo usar backtracking, errores frecuentes y material de apoyo

**¿Cuándo usarlo?**
- Necesitas **todas** las soluciones posibles, o
- Las restricciones pueden evaluarse **incrementalmente** (poda temprana real), o
- El espacio es exponencial pero **podable**.

**¿Cuándo evitarlo?**
- Solo necesitas una solución y hay heurísticas o un algoritmo *greedy* que funciona.
- Hay mucho solapamiento de subproblemas — usa programación dinámica en su lugar.
- No puedes podar efectivamente el espacio de búsqueda (entonces es solo fuerza bruta con pasos extra).

**Errores frecuentes**
- Olvidar el `pop()` (o el `remove`/`deshacer` correspondiente) — sin retroceder, el estado queda contaminado para las
  siguientes ramas.
- Guardar una **referencia** a la solución en vez de una **copia** (`resultado.append(solucion)` en vez de
  `resultado.append(solucion[:])`) — todas las soluciones guardadas terminan apuntando a la misma lista, vacía al final.
- Validar demasiado tarde: revisar la restricción **antes** de avanzar (poda), no después de construir la solución completa.
- No definir una condición de parada alcanzable, o no progresar hacia ella.

**Material de apoyo**
- [Fundamentos del Backtracking (PDF)](_static/unidad2/backtracking/FundamentosdelBacktracking.pdf)

> El backtracking es la culminación natural de la recursión: la misma estructura de caso base + caso recursivo, ahora con
> una decisión explícita en cada paso y la disciplina de deshacer lo que no funcionó.
